# TP01 - Première utilisation des données

In [2]:
from os import name

import pandas as pd
import duckdb
import pyarrow.parquet as pq
import numpy as np

file = "../data/food.parquet"

batch_size = 70000

### 3. Premier dataset et caractéristiques

#### Chargement du dataset complet

In [ ]:
# Chargement complet avec PyArrow (à éviter)
# fichier_charge = pq.ParquetFile(file)
# dt = pd.DataFrame()
# for batch in fichier_charge.iter_batches(batch_size=batch_size):
#     temp_df = batch.to_pandas()
#     dt = pd.concat([dt,temp_df])
# len(dt)

In [3]:
# Affichage des colonnes sans charger le fichier
rel = duckdb.sql(f"SELECT * FROM '{file}' limit 0")
rel.columns

['additives_n',
 'additives_tags',
 'allergens_tags',
 'brands_tags',
 'brands',
 'categories',
 'categories_tags',
 'categories_properties',
 'checkers_tags',
 'ciqual_food_name_tags',
 'cities_tags',
 'code',
 'compared_to_category',
 'complete',
 'completeness',
 'correctors_tags',
 'countries_tags',
 'created_t',
 'creator',
 'data_quality_errors_tags',
 'data_quality_info_tags',
 'data_quality_warnings_tags',
 'data_sources_tags',
 'environmental_score_data',
 'environmental_score_grade',
 'environmental_score_score',
 'environmental_score_tags',
 'editors',
 'emb_codes_tags',
 'emb_codes',
 'entry_dates_tags',
 'food_groups_tags',
 'generic_name',
 'images',
 'informers_tags',
 'ingredients_analysis_tags',
 'ingredients_from_palm_oil_n',
 'ingredients_n',
 'ingredients_original_tags',
 'ingredients_percent_analysis',
 'ingredients_tags',
 'ingredients_text',
 'ingredients_with_specified_percent_n',
 'ingredients_with_unspecified_percent_n',
 'ingredients_without_ciqual_codes_n',


#### Caractéristiques :

- Dimensions : (4636471, 111)
- Types : bool(1), float32(1), float64(25), int32(1), object(56), str(27)
- Utilisation mémoire : 21.8+ GB de ram
- Taux de remplissage par colonne : voir le code

In [8]:
# print(f"Dimensions : {dt.shape}")
# print(f"Informations globales : \n")
# dt.info()
# print(f"Taux de remplissage par colonne :")
# dt.notna().mean()*100

In [12]:
remplissage_col = {}

for column in np.array_split(np.array(rel.columns),5)[0:1] :
    temp_df = pd.read_parquet(file, columns=column.tolist())
    taux = temp_df.notna().mean()
    remplissage_col.update(taux.to_dict())
    del temp_df

print(f"Taux de remplissage par colonne : {pd.Series(remplissage_col).mean()*100:.2f} %")

Taux de remplissage par colonne : 75.72 %


### 4. Cinq questions à résoudre

- combien de produits vendus en France ?
- quelle part a un Nutri-Score renseigné ?
- les dix marques les plus présentes ?
- le taux de manquants sur les nutriments clés ( energy_100g ,sugars_100g , salt_100g ) ?
- qu'est-ce qui vous semble le plus « sale » dans ces données ?

#### Chargement d'un dataset filtré

> Pour répondre à ces questions, on a fait le choix de sélectionner une version filtré du dataset, avec seulement les colonnes qui nous intéressent.

Chargement des données avec PyArrow

In [9]:
fichier_charge = pq.ParquetFile(file)
df_france = pd.DataFrame()
for batch in fichier_charge.iter_batches(batch_size=batch_size, columns=["product_name", "categories", "brands", "countries_tags", "nutriscore_score", "nutriscore_grade", "nutriments"]):
    temp_df = batch.to_pandas()
    temp_df = temp_df[temp_df["countries_tags"].apply(lambda x : isinstance(x, np.ndarray) and "en:france" in x)]
    df_france = pd.concat([df_france,temp_df])

Chargement des données avec DuckDB

In [10]:
#Chargement avec duckdb
# df_france = duckdb.sql("""
#     SELECT "product_name", "categories", "brands", "countries_tags", "nutriscore_score", "nutriscore_grade", "nutriments"
#     FROM '../data/food.parquet'
#     WHERE list_contains(countries_tags, 'en:france')
# """).df()

Chargement des données avec Pandas

In [11]:
#Chargement avec Pandas
# dt = pd.read_parquet(file, columns=["product_name", "categories", "brands", "countries_tags", "nutriscore_score", "nutriscore_grade", "nutriments"])
# df_france = dt[dt["countries_tags"].apply(lambda x : isinstance(x, np.ndarray) and "en:france" in x)]
# len(df_france)

### Combien de produits vendus en France ?

Il y a 1 254 128 produits vendus en France

In [12]:
len(df_france)

1254128

### Quelle part a un Nutri-Score renseigné ?

Il y a 466 032 produits vendus en France avec un nutri-score. Ceux qui ne sont pas éligibles ne sont pas inclus.

In [13]:
df_france[df_france["nutriscore_grade"].isin(["a","b","c","d","e"])]["nutriscore_grade"].count()
#df_france.groupby("nutriscore_grade")["nutriscore_grade"].count()

np.int64(466032)

### Les dix marques les plus présentes ?

Elles sont : Carrefour, U, Auchan, Leader Price, Casino, Cora, Le Gaulois, Picard, Monoprix et Nestlé

In [14]:
df_france.groupby("brands")["brands"].count().sort_values(ascending=False)[1:11]

brands
Carrefour       12009
U               11987
Auchan           6378
Leader Price     5431
Casino           5160
Cora             3960
Le Gaulois       3547
Picard           3504
Monoprix         3403
Nestlé           3348
Name: brands, dtype: int64

### Le taux de manquants sur les nutriments clés ( energy_100g ,sugars_100g , salt_100g ) ?

Taux de manquants :
- Sucre : 29.1 %
- Sel : 33.5 %
- Énergie : 28.5 %

In [15]:
cpt_energy, cpt_sugars, cpt_salt = 0,0,0
total = len(df_france)

for e in df_france["nutriments"] :
    if e is None :
        continue
    for nutriment in e :
        name = nutriment.get("name")
        if name == "energy" : cpt_energy +=1
        if name == "sugars" : cpt_sugars +=1
        if name == "salt" : cpt_salt +=1

In [16]:
print(f"Taux de sucre : {(total-cpt_sugars)/total*100:.1f} %")
print(f"Taux de sel : {(total-cpt_salt)/total*100:.1f} %")
print(f"Taux d'énergie : {(total-cpt_energy)/total*100:.1f} %")

Taux de sucre : 29.1 %
Taux de sel : 33.5 %
Taux d'énergie : 28.5 %


### Qu'est-ce qui vous semble le plus « sale » dans ces données ?

Les dictionnaires stockés dans les colonnes sont plutôt difficiles à gérer, et on n'est jamais sûr de trouver la donnée qu'on cherche. Il y a beaucoup de données vides. Les intitulés des colonnes sont aussi très peu explicites et diffèrent entre les différents jeux de données. Beaucoup de données sont aussi dupliquées (pas de clés étrangères, catégories sous forme de texte, etc.).

### Optimisation du dataset

Colonnes à optimiser :
- nutriscore_grade : score entre a et e
- nutriscore_score : score entre -17 et 57
- categories : 120 000 catégories sur 1 400 000 lignes

In [17]:
def optimise_col_category(df, col_name) :
    print(df[col_name].memory_usage(deep=True)/1000000,"Mo")
    df[col_name] = df[col_name].astype("category")
    print(df[col_name].memory_usage(deep=True)/1000000,"Mo")
    return df

In [18]:
df_france = optimise_col_category(df_france,"categories")

80.343463 Mo
30.639637 Mo


In [19]:
df_france = optimise_col_category(df_france,"nutriscore_grade")

26.319664 Mo
11.287235 Mo


In [20]:
#Nutriscore en int8
print(df_france["nutriscore_score"].memory_usage(deep=True)/1000000,"Mo")
df_france["nutriscore_score"] = df_france['nutriscore_score'].astype("Int8")
print(df_france["nutriscore_score"].memory_usage(deep=True)/1000000,"Mo")

20.066048 Mo
12.54128 Mo
